# ── 0. 配置区 ──────────────────────────────────────────────

In [ ]:
# ── 0. 配置区 ──────────────────────────────────────────────
from pathlib import Path

version   = "sahi_null_v3"
GT_FIELD  = "ground_truth"
IOU_THR   = 0.50
FILENAME_YEAR = 2024

confidence_thresholds = [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.91, 0.92, 0.93]

model_root = Path("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/02_fine-tuned_checkpoint/best_models/04_swd_hbb/null_image_trained_final_checkpoint")
model_names = [
    "yolo11x_20pct_null_images_add_rawData_batch_8_final.pt",
    "yolo11m_20pct_null_images_add_rawData_batch_16_final.pt",
    "yolo11m_20pct_null_images_add_rawData_batch_8_final.pt",
    "yolo11l_20pct_null_images_add_rawData_batch_4_final.pt",
    "yolo11s_20pct_null_images_add_rawData_batch_4_final.pt",
    "yolo11n_20pct_null_images_add_rawData_batch_4_final.pt",
    "yolo11x_20pct_null_images_add_rawData_batch_4_final.pt",
    "yolo11s_20pct_null_images_add_rawData_batch_16_final.pt",
    "yolo11n_20pct_null_images_add_rawData_batch_16_final.pt",
    "yolo11n_20pct_null_images_add_rawData_batch_8_final.pt",
    "yolo11l_20pct_null_images_add_rawData_batch_16_final.pt",
    "yolo11l_20pct_null_images_add_rawData_batch_8_final.pt",
    "yolo11s_20pct_null_images_add_rawData_batch_8_final.pt",
    "yolo11m_20pct_null_images_add_rawData_batch_4_final.pt",
]
ckpt_paths = [str(model_root / name) for name in model_names]

data_root          = Path("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/01_data/a02_16mp_2024_datasets_fiftyone")
target_subdir_name = Path("data")

out_dir = Path("./_eval_exports_per_images") / version
out_dir.mkdir(parents=True, exist_ok=True)

# ── 1. 初始化日志 ──────────────────────────────────────────

In [ ]:
import logging
import ipynbname

log_file_name = ipynbname.name() + ".log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("============ Notebook logging 初始化完成 ============")

# ── 2. Imports ──────────────────────────────────────────

In [ ]:
from __future__ import annotations
from typing import Dict, Any, List, Tuple
from functools import partial

import pandas as pd
import fiftyone as fo
from fiftyone import ViewField as F

from ty_eval_tools import (
    parse_dt_focus,
    export_image_level_rows,
    summarize_from_image_df,
)


def fetch_subsequent_dir(root: Path, subdir_name: Path) -> Tuple[List[Path], List[str]]:
    data_paths = list(root.glob(f"*/{subdir_name}"))
    subdir_path_list = [p.parent for p in data_paths]
    subdir_name_list = [d.name for d in subdir_path_list]
    return subdir_path_list, subdir_name_list


def model_tag_from_path(p: str) -> str:
    return Path(p).stem


def safe_slug(s: str) -> str:
    return s.replace("/", "_").replace(" ", "_").replace("-", "__")


# 绑定年份参数，让 parse_dt_focus_fn 只需接收 filepath
parse_dt_focus_fn = partial(parse_dt_focus, year=FILENAME_YEAR)

# ── 3. 获取批量路径（Master 确认后继续）────────────────

In [ ]:
# ── Step 1：扫描数据子目录，打印路径供 Master 确认 ─────────
logger.info("Step 1 开始：扫描数据子目录")

subdir_path_list, subdir_name_list = fetch_subsequent_dir(data_root, target_subdir_name)
print(f"Found subdirs: {len(subdir_name_list)}")
for n, p in zip(subdir_name_list, subdir_path_list):
    print(f"  {n}  ->  {p}")

logger.info(f"Step 1 完成：共 {len(subdir_name_list)} 个子目录")

# ── 4. Step 2：批量 Per-Image 评估 + 导出 ──────────────

In [ ]:
# ── Step 2：对每个 dataset × 每个模型 × 每个阈值跑逐图评估 ──
# 输入：FiftyOne datasets（含 GT 和各模型 pred_field）
# 输出：per-image CSV + 全局 summary CSV

logger.info("Step 2 开始：批量 per-image 评估")

summary_rows: List[Dict[str, Any]] = []

for subdir_path, subdir_name in zip(subdir_path_list, subdir_name_list):
    dataset_name = f"{version}_{subdir_name}"
    try:
        ds = fo.load_dataset(dataset_name)
    except Exception as e:
        logger.error(f"加载 dataset 失败 {dataset_name}: {e}")
        continue

    logger.info(f"处理: {dataset_name}")
    ds.delete_evaluations()

    for ckpt_path in ckpt_paths:
        model_tag = model_tag_from_path(ckpt_path)
        pred_field = f"small_slices_{model_tag}"

        if pred_field not in ds.get_field_schema():
            logger.warning(f"Missing pred_field={pred_field} in {dataset_name}, skip.")
            continue

        for thr in confidence_thresholds:
            view = ds.filter_labels(pred_field, F("confidence") > thr, only_matches=False)
            eval_key = f"eval_img__{safe_slug(model_tag)}__c{int(thr*100)}__iou{int(IOU_THR*100)}"

            try:
                view.evaluate_detections(
                    pred_field, gt_field=GT_FIELD, eval_key=eval_key,
                    iou=IOU_THR, compute_mAP=False,
                )
            except Exception as e:
                logger.error(f"evaluate_detections 失败 {eval_key}: {e}")
                continue

            try:
                img_df = export_image_level_rows(
                    view=view,
                    dataset_name=dataset_name,
                    subdir_name=subdir_name,
                    subdir_path=str(subdir_path),
                    model_tag=model_tag,
                    ckpt_path=ckpt_path,
                    pred_field=pred_field,
                    conf_thr=thr,
                    eval_key=eval_key,
                    version=version,
                    iou_thr=IOU_THR,
                    gt_field=GT_FIELD,
                    parse_dt_focus_fn=parse_dt_focus_fn,
                )
            except Exception as e:
                logger.error(f"export_image_level_rows 失败 {eval_key}: {e}")
                continue

            out_img = (
                out_dir / safe_slug(dataset_name) / safe_slug(model_tag)
                / f"image_level_{safe_slug(model_tag)}__c{int(thr*100)}__iou{int(IOU_THR*100)}.csv"
            )
            out_img.parent.mkdir(parents=True, exist_ok=True)
            try:
                img_df.to_csv(out_img, index=False)
                logger.info(f"[SAVE] {out_img}")
            except Exception as e:
                logger.error(f"保存 CSV 失败 {out_img}: {e}")

            srow = {
                "dataset_name": dataset_name, "subdir_name": subdir_name,
                "subdir_path": str(subdir_path), "version": version,
                "model_tag": model_tag, "ckpt_path": ckpt_path,
                "pred_field": pred_field, "confidence_threshold": thr, "iou_threshold": IOU_THR,
            }
            srow.update(summarize_from_image_df(img_df))
            summary_rows.append(srow)

summary_df = pd.DataFrame(summary_rows)
out_sum = out_dir / f"per_image_summary__{version}__iou{int(IOU_THR*100)}.csv"
try:
    summary_df.to_csv(out_sum, index=False)
    logger.info(f"已保存汇总: {out_sum}, shape={summary_df.shape}")
except Exception as e:
    logger.error(f"保存汇总 CSV 失败: {e}")

logger.info("Step 2 完成：per-image 评估结束")

# ── 5. 验证：抽查结果 ────────────────────────────────────

In [ ]:
# ── 验证：展示 summary_df 前几行 ──────────────────────────
logger.info(f"验证：summary_df shape={summary_df.shape}")
display(summary_df.head(10))